# 02 — Generate the scientific problem-solving GRPO dataset

This notebook builds a separate task bank for GRPO. It excludes every paper used
for SFT, generates task-only prompts plus hidden task-specific rubrics, and
demonstrates the exact reward:

\[
R = F\times(0.10 + 0.90J)
\]

`F` is an exact four-section format gate. `J` is the average of four integer
0–4 scores from `gpt-5.6-luna`: brainstorm, principles, synthesis, and answer.

## Configuration

Dataset-generation and reward parameters are explicit below. `OPENAI_API_KEY`
remains an environment secret; Hugging Face uses cached login unless the optional
token line is uncommented.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import JSON, Markdown, display

from science_course.data import (
    TASK_FAMILIES,
    build_grpo_dataset,
    read_jsonl,
    render_completion,
    stream_open_science_sources,
    task_quota_status,
    write_jsonl,
)
from science_course.hub import require_hf_namespace
from science_course.judge import (
    DEFAULT_JUDGE_MODEL,
    ScientificDesignJudge,
)
from science_course.rewards import (
    combined_reward,
    format_reward,
    semantic_score_from_judgment,
)
from science_course.teacher import (
    DEFAULT_CRITIC_MODEL,
    DEFAULT_TEACHER_MODEL,
    generate_canonical_tasks,
    require_openai_key,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"

# Local artifacts
SFT_CANONICAL = DATA / "canonical" / "scientific_design_sft_tasks.jsonl"
RAW_SOURCES = DATA / "raw" / "scientific_design_grpo_sources.jsonl"
ACCEPTED = DATA / "canonical" / "scientific_design_grpo_tasks.jsonl"
REJECTED = DATA / "canonical" / "scientific_design_grpo_rejected.jsonl"
GRPO_DISK = DATA / "processed" / "scientific_design_grpo"
JUDGE_CACHE = ROOT / "results" / "scientific_design_judge_cache.jsonl"

# Teacher, critic, and GRPO judge
TEACHER_MODEL = DEFAULT_TEACHER_MODEL
CRITIC_MODEL = DEFAULT_CRITIC_MODEL
JUDGE_MODEL = DEFAULT_JUDGE_MODEL

# Exact accepted-example targets
GRPO_SPLIT_TARGETS = {
    "grpo_train": 500,
    "grpo_validation": 75,
    "test": 100,
}
TASK_FAMILY_WEIGHTS = {
    "mechanism_guided_design": 0.25,
    "experimental_design": 0.25,
    "troubleshooting": 0.20,
    "hypothesis_development": 0.15,
    "cross_domain_synthesis": 0.15,
}

# Open scientific source pool
SOURCE_DATASET_ID = "common-pile/peS2o"
SOURCE_SPLIT = "train"
SOURCE_CANDIDATE_LIMIT = 3_000
MAX_SOURCE_RECORDS_SCANNED = 250_000
MIN_SOURCE_CHARS = 1_200
MAX_SOURCE_CHARS = 6_000
RANDOM_SEED = 29

# Resumable API generation
GENERATION_CONCURRENCY = 8
MAX_GENERATION_ATTEMPTS = 3_000
OPENAI_MAX_RETRIES = 3
OPENAI_TIMEOUT_SECONDS = 120.0

# Exact GRPO reward coefficients
FORMAT_BASE_REWARD = 0.10
SEMANTIC_REWARD_WEIGHT = 0.90
JUDGE_DIMENSION_MAX_SCORE = 4

# Hugging Face publication
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
DATASET_CONFIG_NAME = "scientific_design_grpo"
PUSH_DATASETS_TO_HUB = True
DATASET_PRIVATE = False
HF_TOKEN = None
# HF_TOKEN = __import__("os").environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

require_openai_key()
if set(TASK_FAMILY_WEIGHTS) != set(TASK_FAMILIES):
    raise ValueError(f"Define weights for exactly these families: {TASK_FAMILIES}")
if abs(FORMAT_BASE_REWARD + SEMANTIC_REWARD_WEIGHT - 1.0) > 1e-9:
    raise ValueError("The reward coefficients must sum to one.")
if JUDGE_DIMENSION_MAX_SCORE != 4:
    raise ValueError("The tested Luna rubric uses integer scores from 0 to 4.")
if PUSH_DATASETS_TO_HUB:
    require_hf_namespace(DATASET_HF_REPO, token=HF_TOKEN)

sft_canonical = read_jsonl(SFT_CANONICAL)
if not sft_canonical:
    raise RuntimeError("Run notebook 01 first.")
sft_paper_ids = {row["paper_id"] for row in sft_canonical}
sns.set_theme(style="whitegrid", context="talk")
display(
    JSON(
        {
            "teacher_model": TEACHER_MODEL,
            "critic_model": CRITIC_MODEL,
            "judge_model": JUDGE_MODEL,
            "accepted_targets": GRPO_SPLIT_TARGETS,
            "excluded_sft_papers": len(sft_paper_ids),
            "reward_formula": "F * (0.10 + 0.90 * J)",
        }
    )
)

## 1. Acquire a paper-disjoint source pool

Every paper used by SFT is excluded before GRPO task generation. This makes the
two adaptation stages and final test set traceable to disjoint source documents.

In [ ]:
cached_sources = read_jsonl(RAW_SOURCES)
cache_is_usable = (
    len(cached_sources) == SOURCE_CANDIDATE_LIMIT
    and not ({row["paper_id"] for row in cached_sources} & sft_paper_ids)
    and all(
        row.get("source_dataset") == SOURCE_DATASET_ID
        and row.get("source_split") == SOURCE_SPLIT
        for row in cached_sources
    )
)
if cache_is_usable:
    sources = cached_sources
    print(f"Reusing {len(sources)} cached GRPO sources")
else:
    sources = stream_open_science_sources(
        dataset_id=SOURCE_DATASET_ID,
        split=SOURCE_SPLIT,
        max_papers=SOURCE_CANDIDATE_LIMIT,
        max_scanned=MAX_SOURCE_RECORDS_SCANNED,
        min_chars=MIN_SOURCE_CHARS,
        max_chars=MAX_SOURCE_CHARS,
        seed=RANDOM_SEED,
        exclude_paper_ids=sft_paper_ids,
    )
    write_jsonl(RAW_SOURCES, sources)
    print(f"Cached {len(sources)} paper-disjoint source candidates")
if {row["paper_id"] for row in sources} & sft_paper_ids:
    raise RuntimeError("SFT/GRPO source-paper leakage detected.")

## 2. Generate GRPO tasks and hidden rubrics

GRPO references are retained only in the local canonical audit. The policy-facing
projection contains the task and a hidden rubric for reward calculation—never a
gold completion.

In [ ]:
canonical = generate_canonical_tasks(
    sources,
    accepted_path=ACCEPTED,
    rejected_path=REJECTED,
    split_targets=GRPO_SPLIT_TARGETS,
    task_family_weights=TASK_FAMILY_WEIGHTS,
    teacher_model=TEACHER_MODEL,
    critic_model=CRITIC_MODEL,
    concurrency=GENERATION_CONCURRENCY,
    max_attempts=MAX_GENERATION_ATTEMPTS,
    api_max_retries=OPENAI_MAX_RETRIES,
    api_timeout_seconds=OPENAI_TIMEOUT_SECONDS,
)
status = task_quota_status(
    canonical,
    GRPO_SPLIT_TARGETS,
    TASK_FAMILY_WEIGHTS,
)
if not status["complete"]:
    raise RuntimeError(f"Unfilled GRPO quotas: {status['deficits']}")
print(
    {
        "accepted": len(canonical),
        "rejected": len(read_jsonl(REJECTED)),
        "split_counts": pd.Series(
            [row["split"] for row in canonical]
        ).value_counts().to_dict(),
    }
)

In [ ]:
grpo = build_grpo_dataset(canonical)
expected_splits = {
    "train": GRPO_SPLIT_TARGETS["grpo_train"],
    "validation": GRPO_SPLIT_TARGETS["grpo_validation"],
    "test": GRPO_SPLIT_TARGETS["test"],
}
actual_splits = {name: len(split) for name, split in grpo.items()}
if actual_splits != expected_splits:
    raise RuntimeError(
        f"Unexpected GRPO split sizes: {actual_splits}; expected {expected_splits}"
    )
if GRPO_DISK.exists():
    shutil.rmtree(GRPO_DISK)
grpo.save_to_disk(GRPO_DISK)
for split_name, split_data in grpo.items():
    split_data.to_parquet(
        DATA / "processed" / f"scientific_design_grpo_{split_name}.parquet"
    )
if PUSH_DATASETS_TO_HUB:
    grpo.push_to_hub(
        DATASET_HF_REPO,
        config_name=DATASET_CONFIG_NAME,
        private=DATASET_PRIVATE,
        token=HF_TOKEN,
        commit_message="Publish scientific design GRPO dataset",
    )
for split_data in grpo.values():
    assert "source_text" not in split_data.column_names
    assert "completion" not in split_data.column_names
    assert "answer" not in split_data.column_names
print(grpo)

In [ ]:
row = grpo["train"][0]
display(Markdown("### What the policy sees"))
display(JSON(row["prompt"]))
display(Markdown("### What the reward can see"))
display(
    JSON(
        {
            key: row[key]
            for key in (
                "task",
                "required_constraints",
                "evaluation_criteria",
                "acceptable_alternatives",
                "failure_modes",
            )
        }
    )
)

## 3. Demonstrate the exact reward

A malformed response gets `F=0` and therefore total reward zero without an API
call. A valid response gets the 0.10 format base plus 0.90 times Luna's semantic
score. Luna grades all four dimensions from 0 to 4 and accepts valid alternatives.

In [ ]:
canonical_by_id = {item["task_id"]: item for item in canonical}
reference_record = canonical_by_id[row["task_id"]]
reference_completion = render_completion(reference_record)
format_score = format_reward([reference_completion])[0]

judge_item = {
    "task": row["task"],
    "required_constraints": row["required_constraints"],
    "evaluation_criteria": row["evaluation_criteria"],
    "acceptable_alternatives": row["acceptable_alternatives"],
    "failure_modes": row["failure_modes"],
    "student_response": reference_completion,
}
judge = ScientificDesignJudge(
    model=JUDGE_MODEL,
    cache_path=JUDGE_CACHE,
    api_max_retries=OPENAI_MAX_RETRIES,
    api_timeout_seconds=OPENAI_TIMEOUT_SECONDS,
)
judgment = judge.judgments([judge_item])[0]
semantic_score = semantic_score_from_judgment(judgment)
total_reward = combined_reward(
    format_score,
    semantic_score,
    format_base_reward=FORMAT_BASE_REWARD,
    semantic_reward_weight=SEMANTIC_REWARD_WEIGHT,
)
display(
    JSON(
        {
            "format_score_F": format_score,
            "luna_judgment": judgment,
            "semantic_score_J": semantic_score,
            "total_reward_R": total_reward,
        }
    )
)

## 4. Audit the GRPO curriculum

The following plots verify family balance, task sizes, and rubric depth. The
final test split remains untouched by either SFT or GRPO updates.

In [ ]:
frame = pd.DataFrame(
    [
        {
            "split": split_name,
            "task_family": item["task_family"],
            "task_chars": len(item["task"]),
            "constraints": len(item["required_constraints"]),
            "criteria": len(item["evaluation_criteria"]),
        }
        for split_name, split_data in grpo.items()
        for item in split_data
    ]
)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(
    frame,
    x="task_family",
    hue="split",
    ax=axes[0],
)
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_title("GRPO task-family balance")
sns.scatterplot(
    frame,
    x="constraints",
    y="criteria",
    hue="task_family",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Hidden rubric depth")
plt.tight_layout()
plt.show()

## Result

The default GRPO dataset contains 500 training, 75 validation, and 100 final-test
tasks. Notebook 03 trains the four-section SFT policy; notebook 04 then uses the
single combined Luna reward demonstrated above.